In [ ]:
#| default_exp compute

In [ ]:
#| include: false
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import warnings
from dataclasses import dataclass

import torch
import torch.nn as nn

try:
    from thop import profile as _thop_profile
except ImportError:
    _thop_profile = None

try:
    from torchprofile import profile_macs as _profile_macs
except ImportError:
    _profile_macs = None

In [ ]:
#| export
@dataclass(slots=True)
class ComputeMetrics:
    """MACs (Multiply-Accumulate operations) in millions."""
    macs_m: float | None  # None if unavailable

    @property
    def macs_available(self) -> bool:
        """Check if MACs measurement succeeded."""
        return self.macs_m is not None

    def as_dict(self) -> dict[str, float]:
        return {
            "macs_m": self.macs_m if self.macs_m is not None else float("nan"),
        }


#| export
def compute_compute(
    model: nn.Module,        # model to analyze
    sample: torch.Tensor,    # input tensor (with batch dimension)
) -> ComputeMetrics:
    """Compute MACs for a single forward pass."""
    try:
        model_device = next(model.parameters()).device
    except StopIteration:
        model_device = torch.device("cpu")

    if sample.device != model_device:
        sample = sample.to(model_device)

    macs_m: float | None = None

    if _thop_profile is not None:
        try:
            mac_raw, _ = _thop_profile(model, inputs=(sample,), verbose=False)
            macs_m = round(mac_raw / 1e6, 3)
        except Exception as e:
            warnings.warn(f"thop failed: {e}")
    elif _profile_macs is not None:
        try:
            macs_m = round(_profile_macs(model, sample) / 1e6, 3)
        except Exception as e:
            warnings.warn(f"torchprofile failed: {e}")
    else:
        warnings.warn("No MAC-counting backend available – skipping MACs")

    return ComputeMetrics(macs_m=macs_m)

In [ ]:
show_doc(ComputeMetrics)

In [ ]:
show_doc(compute_compute)

In [ ]:
#| hide
from fastcore.test import *

import torch, torch.nn as nn
_m = nn.Linear(10, 5)
_x = torch.randn(1, 10)
_c = compute_compute(_m, _x)
assert isinstance(_c, ComputeMetrics)

---

## See Also

- [Size](size.html) — Model size measurement
- [Benchmark](../analysis/benchmark.html) — Unified API